Save base Logit, KNN, SVC, LogisticRegression, CatBoost, XGboost with pipeline. Sedimentation_Stk feature was used!

Based on `research_14_SMOTE_influence_on_metrics.ipynb` it was decided to use SMOTE for all models, except Logit!

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

import optuna
from functools import partial

from pathlib import Path
import sys

from typing import Type
from sklearn.base import BaseEstimator

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    StatsModelsEstimator,
    update_estimator_params,
    OptunaOptimizer,
    GridSearchOptimizer,
)

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'opt-model'
add_smote = False
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}
scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
n_trials = 1000 # was checked with 1000 and 5000

save_model_and_metrics = False

## Optimization functions

In [5]:
def optimize_estimator_optuna(
    target:str,
    estimator:Type[BaseEstimator],
    estimator_params:dict,
    objective:callable,
    n_trials:int,
    step_scoring_average:str=step_scoring_average,
    params_to_process:list=None,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=save_model_and_metrics,
):
    
    # Switch off probability for SVC, since it is long to optimize
    if 'probability' in estimator_params:
        estimator_params['probability'] = False
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics,
        step_scoring_average=step_scoring_average,
    )
    
    opt = OptunaOptimizer(
        objective=partial(objective, ml_pipe=ml_pipe),
        study_name="model_study",
        direction="maximize",
    )
    
    opt.optimize(n_trials=n_trials)
    
    best_params = opt.study.best_params
    
    if params_to_process:
        for param in params_to_process:
            # Find one key in best_params which contains param
            key = next((k for k in best_params.keys() if param in k), None)
            if key:
                best_params[param] = best_params.pop(key)

    print('best_params')
    display(best_params)
    
    # Switch on probability for SVC, to get metrics like roc_auc for the final model
    if 'probability' in estimator_params:
        estimator_params['probability'] = True
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params={**estimator_params, **best_params},
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        step_scoring_average=step_scoring_average, # No need to pass, it would not be used
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

# Logistic Regression

In [6]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
    'max_iter': 1000,
}
params_to_process = ['penalty']

def logreg_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
    
    # # NOTE: Does not work, since optuna requires static parameters ranges
    # solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'])
    # penalty_options = {
    #     'lbfgs': ['l2'],
    #     'liblinear': ['l1', 'l2'],
    #     'newton-cg': ['l2'],
    #     'newton-cholesky': ['l2'],
    #     'sag': ['l2'],
    #     'saga': ['l1', 'l2', 'elasticnet'],
    # }
    # penalty = trial.suggest_categorical('penalty', penalty_options[solver])
    # NOTE: warnings + inconvenient to create best model
    # solver_penalty_combinations = [
    #     ("liblinear", "l1"),
    #     ("liblinear", "l2"),
    #     ("saga", "l1"),
    #     ("saga", "l2"),
    #     ("saga", "elasticnet"),
    #     ("lbfgs", "l2"),
    #     ("newton-cg", "l2"),
    #     ("newton-cholesky", "l2"),
    #     ("sag", "l2"),
    # ]
    # solver, penalty = trial.suggest_categorical(
    #     'solver_penalty', solver_penalty_combinations
    # )
    
    solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'])
    # solver = trial.suggest_categorical('solver', ['lbfgs',])
    
    if solver == 'liblinear':
        penalty = trial.suggest_categorical('penalty_liblinear', ['l1', 'l2'])
    elif solver == 'saga':
        penalty = trial.suggest_categorical('penalty_saga', ['l1', 'l2', 'elasticnet'])
    else:
        penalty = trial.suggest_categorical('penalty_other', ['l2'])
    
    # If C is large, the penalty should be small
    C = trial.suggest_float('C', 1e-4, 1e3, log=True)
    
    l1_ratio = None
    if penalty == 'elasticnet':
        l1_ratio = trial.suggest_float('l1_ratio', 0., 1.)
    
    class_weight = trial.suggest_categorical(
        'class_weight', [None, 'balanced']
    )
    
    suggested_estimator_params = {
        "solver": solver,
        "penalty": penalty,
        "C": C,
        "l1_ratio": l1_ratio,
        "class_weight": class_weight,
    }
    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=logreg_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-14 22:06:13,851] A new study created in memory with name: model_study
[I 2025-04-14 22:06:13,907] Trial 0 finished with value: 0.7985639090256361 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l2', 'C': 1.6136341713591311, 'class_weight': None}. Best is trial 0 with value: 0.7985639090256361.
[I 2025-04-14 22:06:13,966] Trial 1 finished with value: 0.7941608217143109 and parameters: {'solver': 'lbfgs', 'penalty_other': 'l2', 'C': 0.47129737561107793, 'class_weight': None}. Best is trial 0 with value: 0.7985639090256361.
[I 2025-04-14 22:06:14,018] Trial 2 finished with value: 0.26121337827326935 and parameters: {'solver': 'saga', 'penalty_saga': 'elasticnet', 'C': 0.00021142332035497166, 'l1_ratio': 0.6075448519014384, 'class_weight': None}. Best is trial 0 with value: 0.7985639090256361.
[I 2025-04-14 22:06:14,072] Trial 3 finished with value: 0.797286337918985 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l1', 'C': 0.292575779498244, 'class_w

KeyboardInterrupt: 

In [7]:
target = 'no_fragmentation'
estimator_params = {
    "fit_intercept": False,
    'max_iter': 1000,
}

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=logreg_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-14 21:43:17,913] A new study created in memory with name: model_study
[I 2025-04-14 21:43:17,962] Trial 0 finished with value: 0.8397856760138982 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l2', 'C': 1.6136341713591311, 'class_weight': None}. Best is trial 0 with value: 0.8397856760138982.
[I 2025-04-14 21:43:18,012] Trial 1 finished with value: 0.8027588628638668 and parameters: {'solver': 'lbfgs', 'penalty_other': 'l2', 'C': 0.47129737561107793, 'class_weight': None}. Best is trial 0 with value: 0.8397856760138982.
[I 2025-04-14 21:43:18,056] Trial 2 finished with value: 0.4233049487844008 and parameters: {'solver': 'saga', 'penalty_saga': 'elasticnet', 'C': 0.00021142332035497166, 'l1_ratio': 0.6075448519014384, 'class_weight': None}. Best is trial 0 with value: 0.8397856760138982.
[I 2025-04-14 21:43:18,102] Trial 3 finished with value: 0.7911634439518859 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l1', 'C': 0.292575779498244, 'class_w

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LogisticRegression"


,0
target,no_fragmentation
model,LogisticRegression_no_fragmentation_opt-model
holdout_test_f1_macro,0.798595
holdout_test_accuracy_balanced,0.834091
holdout_test_roc_auc,0.943636
holdout_test_f1,0.723404
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.865709
cv_test_accuracy_balanced_median,0.882051
cv_test_roc_auc_median,0.938462


# KNClassifier

In [ ]:
estimator = KNeighborsClassifier
target = 'splashing'
estimator_params = {}
# params_to_process = ['penalty']
params_to_process = None


def knn_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):

    n_neighbors = trial.suggest_int("n_neighbors", 1, 50)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    p = trial.suggest_int("p", 1, 2)  # 1=manhattan, 2=euclidean
    # leaf_size - affects the speed of the tree construction and search
    # algorithm - affects the speed of the search, for small datasets it is okay to use auto
    
    suggested_estimator_params = {
        "n_neighbors": n_neighbors,
        "weights": weights,
        "p": p,
    }
    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=knn_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-14 21:44:14,171] A new study created in memory with name: model_study
[I 2025-04-14 21:44:14,266] Trial 0 finished with value: 0.7712834391009027 and parameters: {'n_neighbors': 19, 'weights': 'uniform', 'p': 2}. Best is trial 0 with value: 0.7712834391009027.
[I 2025-04-14 21:44:14,353] Trial 1 finished with value: 0.8008110983626221 and parameters: {'n_neighbors': 8, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.8008110983626221.
[I 2025-04-14 21:44:14,444] Trial 2 finished with value: 0.73226117633208 and parameters: {'n_neighbors': 31, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.8008110983626221.
[I 2025-04-14 21:44:14,538] Trial 3 finished with value: 0.7563818520550063 and parameters: {'n_neighbors': 42, 'weights': 'uniform', 'p': 1}. Best is trial 1 with value: 0.8008110983626221.
[I 2025-04-14 21:44:14,627] Trial 4 finished with value: 0.7848640826258931 and parameters: {'n_neighbors': 16, 'weights': 'uniform', 'p': 1}. Best is trial 1

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "KNeighborsClassifier"


,0
target,splashing
model,KNeighborsClassifier_splashing_opt-model
holdout_test_f1_macro,0.836601
holdout_test_accuracy_balanced,0.828704
holdout_test_roc_auc,0.881173
holdout_test_f1,0.888889
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.876935
cv_test_accuracy_balanced_median,0.876935
cv_test_roc_auc_median,0.944361


In [ ]:
estimator = KNeighborsClassifier
target = 'no_fragmentation'
estimator_params = {}
params_to_process = None

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=knn_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-14 21:45:22,424] A new study created in memory with name: model_study
[I 2025-04-14 21:45:22,507] Trial 0 finished with value: 0.7701654488774984 and parameters: {'n_neighbors': 19, 'weights': 'uniform', 'p': 2}. Best is trial 0 with value: 0.7701654488774984.
[I 2025-04-14 21:45:22,586] Trial 1 finished with value: 0.8156628959108977 and parameters: {'n_neighbors': 8, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.8156628959108977.
[I 2025-04-14 21:45:22,670] Trial 2 finished with value: 0.7104448022241227 and parameters: {'n_neighbors': 31, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.8156628959108977.
[I 2025-04-14 21:45:22,755] Trial 3 finished with value: 0.7299310684038487 and parameters: {'n_neighbors': 42, 'weights': 'uniform', 'p': 1}. Best is trial 1 with value: 0.8156628959108977.
[I 2025-04-14 21:45:22,834] Trial 4 finished with value: 0.7481827842300584 and parameters: {'n_neighbors': 16, 'weights': 'uniform', 'p': 1}. Best is trial

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "KNeighborsClassifier"


,0
target,no_fragmentation
model,KNeighborsClassifier_no_fragmentation_opt-model
holdout_test_f1_macro,0.890351
holdout_test_accuracy_balanced,0.865909
holdout_test_roc_auc,0.972727
holdout_test_f1,0.833333
holdout_test_accuracy,0.92
cv_test_f1_macro_median,0.833856
cv_test_accuracy_balanced_median,0.841575
cv_test_roc_auc_median,0.93956


# SVC

In [6]:
estimator = SVC
target = 'splashing'
estimator_params = {
    'probability': True, # Would beFalse for optimization, True for metrics
    # 'shrinking': True, # already in the default params
}
params_to_process = ['gamma']


def svc_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
    kernel = trial.suggest_categorical(
        'kernel', ['linear', 'rbf', 'sigmoid', 'poly']
    )
    C = trial.suggest_float('C', 1e-3, 1e3, log=True)
    
    class_weight = trial.suggest_categorical(
        'class_weight', [None, 'balanced']
    )
    
    suggested_estimator_params = {
        "kernel": kernel,
        "C": C,
        "class_weight": class_weight,
        # Optional params
        # 'gamma': gamma
        # 'coef0': coef0,
        # 'degree': degree,
    }
    
    if kernel in ['rbf', 'poly', 'sigmoid']:
        gamma_choice = trial.suggest_categorical(
            "gamma_type", ["scale", "auto", "float"]
        )
        if gamma_choice == "float":
            gamma = trial.suggest_float("gamma", 1e-4, 10, log=True)
        else:
            gamma = gamma_choice
        suggested_estimator_params['gamma'] = gamma
    
    # Optional params
    if kernel in ['sigmoid', 'poly']:
        suggested_estimator_params['coef0'] = trial.suggest_float(
            "coef0", 0.0, 1.0
        )
    if kernel == 'poly':
        suggested_estimator_params['degree'] = trial.suggest_int(
            "degree", 2, 5
        )
    
    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params
    )
    
    print(estimator_params)
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=svc_objective,
    n_trials=n_trials,
)

In [ ]:
estimator = SVC
target = 'no_fragmentation'
estimator_params = {
    'probability': True, # Would be False for optimization, True for metrics
    # 'shrinking': True, # already in the default params
}
params_to_process = ['gamma']

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=svc_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-14 23:51:59,608] A new study created in memory with name: model_study
[I 2025-04-14 23:51:59,670] Trial 0 finished with value: 0.4233049487844008 and parameters: {'kernel': 'rbf', 'C': 0.008632008168602538, 'class_weight': None, 'gamma_type': 'scale'}. Best is trial 0 with value: 0.4233049487844008.
[I 2025-04-14 23:51:59,741] Trial 1 finished with value: 0.6016942035676301 and parameters: {'kernel': 'rbf', 'C': 0.012329623163659839, 'class_weight': 'balanced', 'gamma_type': 'scale'}. Best is trial 1 with value: 0.6016942035676301.
[I 2025-04-14 23:51:59,792] Trial 2 finished with value: 0.8126333274927476 and parameters: {'kernel': 'linear', 'C': 0.5450293694558254, 'class_weight': None}. Best is trial 2 with value: 0.8126333274927476.


{'probability': False, 'kernel': 'rbf', 'C': 0.008632008168602538, 'class_weight': None, 'gamma': 'scale'}
{'probability': False, 'kernel': 'rbf', 'C': 0.012329623163659839, 'class_weight': 'balanced', 'gamma': 'scale'}
{'probability': False, 'kernel': 'linear', 'C': 0.5450293694558254, 'class_weight': None}
{'probability': False, 'kernel': 'poly', 'C': 0.010547383621352036, 'class_weight': 'balanced', 'gamma': 'scale', 'coef0': 0.09767211400638387, 'degree': 4}


...

{'probability': False, 'kernel': 'poly', 'C': 5.818113224340252, 'class_weight': None, 'gamma': 'auto', 'coef0': 0.9839130335529036, 'degree': 2}
{'probability': False, 'kernel': 'poly', 'C': 13.826246970439835, 'class_weight': None, 'gamma': 'auto', 'coef0': 0.848188873413308, 'degree': 2}
{'probability': False, 'kernel': 'poly', 'C': 10.296217131018246, 'class_weight': None, 'gamma': 'auto', 'coef0': 0.957136274898944, 'degree': 2}
{'probability': False, 'kernel': 'poly', 'C': 15.669617356942982, 'class_weight': None, 'gamma': 'auto', 'coef0': 0.9284153910076138, 'degree': 2}


[I 2025-04-14 23:52:18,012] Trial 300 finished with value: 0.8820456651349314 and parameters: {'kernel': 'poly', 'C': 15.669617356942982, 'class_weight': None, 'gamma_type': 'auto', 'coef0': 0.9284153910076138, 'degree': 2}. Best is trial 14 with value: 0.8870956746492266.
[I 2025-04-14 23:52:18,074] Trial 301 finished with value: 0.8820456651349314 and parameters: {'kernel': 'poly', 'C': 15.23781114692982, 'class_weight': None, 'gamma_type': 'auto', 'coef0': 0.9043586577304755, 'degree': 2}. Best is trial 14 with value: 0.8870956746492266.
[I 2025-04-14 23:52:18,135] Trial 302 finished with value: 0.8820456651349314 and parameters: {'kernel': 'poly', 'C': 16.48550266591709, 'class_weight': None, 'gamma_type': 'auto', 'coef0': 0.8993369780793108, 'degree': 2}. Best is trial 14 with value: 0.8870956746492266.
[I 2025-04-14 23:52:18,196] Trial 303 finished with value: 0.8385239017248766 and parameters: {'kernel': 'poly', 'C': 2.952858254551116, 'class_weight': None, 'gamma_type': 'auto',

{'probability': False, 'kernel': 'poly', 'C': 15.23781114692982, 'class_weight': None, 'gamma': 'auto', 'coef0': 0.9043586577304755, 'degree': 2}
{'probability': False, 'kernel': 'poly', 'C': 16.48550266591709, 'class_weight': None, 'gamma': 'auto', 'coef0': 0.8993369780793108, 'degree': 2}
{'probability': False, 'kernel': 'poly', 'C': 2.952858254551116, 'class_weight': None, 'gamma': 'auto', 'coef0': 0.9023656297073891, 'degree': 2}
{'probability': False, 'kernel': 'poly', 'C': 15.724956833658473, 'class_weight': None, 'gamma': 'auto', 'coef0': 0.8808680061344845, 'degree': 2}


[I 2025-04-14 23:52:18,256] Trial 304 finished with value: 0.8820456651349314 and parameters: {'kernel': 'poly', 'C': 15.724956833658473, 'class_weight': None, 'gamma_type': 'auto', 'coef0': 0.8808680061344845, 'degree': 2}. Best is trial 14 with value: 0.8870956746492266.
[I 2025-04-14 23:52:18,318] Trial 305 finished with value: 0.8820456651349314 and parameters: {'kernel': 'poly', 'C': 14.813066484757773, 'class_weight': None, 'gamma_type': 'auto', 'coef0': 0.8790291685828211, 'degree': 2}. Best is trial 14 with value: 0.8870956746492266.
[I 2025-04-14 23:52:18,378] Trial 306 finished with value: 0.8820456651349314 and parameters: {'kernel': 'poly', 'C': 15.618283680448743, 'class_weight': None, 'gamma_type': 'auto', 'coef0': 0.875313016627218, 'degree': 2}. Best is trial 14 with value: 0.8870956746492266.
[I 2025-04-14 23:52:18,439] Trial 307 finished with value: 0.8820456651349314 and parameters: {'kernel': 'poly', 'C': 15.681425651922781, 'class_weight': None, 'gamma_type': 'auto

{'probability': False, 'kernel': 'poly', 'C': 14.813066484757773, 'class_weight': None, 'gamma': 'auto', 'coef0': 0.8790291685828211, 'degree': 2}
{'probability': False, 'kernel': 'poly', 'C': 15.618283680448743, 'class_weight': None, 'gamma': 'auto', 'coef0': 0.875313016627218, 'degree': 2}
{'probability': False, 'kernel': 'poly', 'C': 15.681425651922781, 'class_weight': None, 'gamma': 'auto', 'coef0': 0.8698375866879595, 'degree': 2}
{'probability': False, 'kernel': 'poly', 'C': 15.877764323826812, 'class_weight': None, 'gamma': 'auto', 'coef0': 0.8605262094022169, 'degree': 2}
